In [5]:
pip install plotly.express

In [12]:
import warnings
import pandas as pd
import plotly.express as px

# Suprimir aviso do openpyxl
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# Carregar os dados
df = pd.read_excel('DADO.xlsx')

# Verificar colunas necessárias
required_columns = ['StartDate', 'EndDate', 'Nome_Unidade', 'Primeiro Classificacao']
if not all(col in df.columns for col in required_columns):
    raise ValueError("Faltam colunas necessárias no arquivo.")

# Limpar dados inválidos nas colunas de data
df = df[df['StartDate'].astype(str).str.contains(r'\d{4}-\d{2}-\d{2}', na=False)]
df = df[df['EndDate'].astype(str).str.contains(r'\d{4}-\d{2}-\d{2}', na=False)]

# Converter colunas de data
df['StartDate'] = pd.to_datetime(df['StartDate'], errors='coerce')
df['EndDate'] = pd.to_datetime(df['EndDate'], errors='coerce')

# Remover linhas com datas inválidas
df = df.dropna(subset=['StartDate', 'EndDate'])

# Calcular duração em dias
df['Duracao_dias'] = (df['EndDate'] - df['StartDate']).dt.days

# Identificar unidade na posição 10
unidade_destacada = df.iloc[10]['Nome_Unidade'] if len(df) > 10 else None

# Criar gráfico de Gantt
fig = px.timeline(
    df,
    x_start='StartDate',
    x_end='EndDate',
    y='Nome_Unidade',
    color='Primeiro Classificacao',
    title='Gráfico de Gantt por Unidade',
    labels={'Nome_Unidade': 'Unidade', 'Primeiro Classificacao': 'Classificação'}
)

# Adicionar duração como anotação (apenas número)
for i, row in df.iterrows():
    fig.add_annotation(
        x=row['StartDate'] + (row['EndDate'] - row['StartDate']) / 2,
        y=row['Nome_Unidade'],
        text=str(row['Duracao_dias']),
        showarrow=False,
        font=dict(size=10, color="black"),
        yshift=10
    )

# Destacar unidade na posição 10
if unidade_destacada:
    fig.update_traces(
        selector=dict(name=unidade_destacada),
        marker=dict(line=dict(width=3, color='black'))
    )

# Ajustar layout
fig.update_layout(
    xaxis_title='Data',
    yaxis_title='Unidade',
    font=dict(size=12),
    plot_bgcolor='white'
)

# Mostrar gráfico
fig.show()
